# Feature Selection & Dimension Reduction - Exercise

In this exercise you will work with the **handwritten digits** dataset that is built into scikit-learn (`load_digits`). Every observation is a small grey-scale image of **8 x 8 pixels** showing a handwritten digit (0-9). Each pixel is one feature with a value between 0 (white) and 16 (black), so every image is described by **64 features** (`pixel_0_0` ... `pixel_7_7`).

The central question of this case: **do we really need all 64 pixels to recognise a digit?**

You will tackle this question in two different ways, just like in class:
1. **Feature Selection** with **Boruta**: keep only the original pixels that are truly relevant.
2. **Dimension Reduction** with **PCA**: replace the 64 pixels by a smaller number of new features (principal components).

For both approaches you will compare the model performance with the performance on all features.

---
---
---
## Complete the tasks in bold

## 0 - Setup

**TASK: Run the cell below to import the libraries you will need.**

In [ ]:
import time  # To time model training
import numpy as np  # To do some calculations
import pandas as pd  # To work with dataframes
import matplotlib.pyplot as plt  # To make plots

from sklearn.datasets import load_digits  # The dataset (built into scikit-learn)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Install missing package in the notebook environment if needed
#%pip install boruta
from boruta import BorutaPy

RANDOM_STATE = 42  # Use this everywhere for reproducible results

## 1 - Data Loading and Exploration

**TASK: Load the digits dataset with `load_digits(return_X_y=True, as_frame=True)` into `X` and `y`. Also store the full dataset object `load_digits()` in a variable called `digits` (it contains the images in `digits.images`). Print the shape of `X` and show the first rows. *(Expected: 1797 rows and 64 columns.)***

In [ ]:
X, y = load_digits(return_X_y=True, as_frame=True)
digits = load_digits()

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
X.head()

**TASK: Check the class distribution of the target `y`. Is the dataset balanced?**

In [ ]:
class_counts = y.value_counts().sort_index()
print("Class distribution of digits (0-9):")
print(class_counts)
print("\nClass percentages:")
print((class_counts / len(y) * 100).round(2))

**TASK: Plot the first 10 images (`digits.images`) in a 2 x 5 grid with `plt.imshow(..., cmap='gray_r')` and use the true label as the title of each image.**

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(f"Label: {y[i]}", fontsize=12, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

**TASK: Calculate the standard deviation of every pixel (`X.std()`), reshape it to 8 x 8 and show it as an image with a colour bar. Also print the names of the pixels that are constant (standard deviation = 0). *(In the solution we found 3 constant pixels.)***

In [ ]:
pixel_std = X.std()
std_image = pixel_std.values.reshape(8, 8)

plt.figure(figsize=(6, 5))
plt.imshow(std_image, cmap='viridis')
plt.colorbar(label='Standard Deviation')
plt.title('Pixel Standard Deviation (8x8 Grid)', fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

constant_pixels = pixel_std[pixel_std == 0].index.tolist()
print(f"Number of constant pixels: {len(constant_pixels)}")
print(f"Constant pixel names: {constant_pixels}")

**QUESTION: Which pixels vary the least? Why does that make sense for images of handwritten digits? Which pixels do you expect a feature selection method to remove?**

*Your answer:*

The outer border pixels, especially the corners (`pixel_0_0`, `pixel_4_0`, and `pixel_4_7`), have a standard deviation of exactly 0 across all 1,797 images.

That makes total sense: people write digits in the center of the 8x8 box. The outer edges and corners are almost always left blank (pixel value 0) no matter which digit was drawn.

Because constant pixels never change from one image to the next, they give the model zero information to separate a 0 from an 8. I expect any feature selection method to throw out these outer border and corner pixels right away.

## 2 - Data Processing

Feature selection (and scaling / PCA) are part of the model: they must be **learned on the training data only**. If we let Boruta or PCA look at the test set, information from the test set leaks into the model and our performance estimate becomes too optimistic (more about this in week 6 - Data Leakage). So we split the data first.

**TASK: Split the data into a training set (75%) and a test set (25%) with `train_test_split`. Use `stratify=y` and `random_state=RANDOM_STATE`. Print the shapes. *(Expected: 1347 training and 450 test observations.)***

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

## 3 - Feature Selection with Boruta

Recall how **Boruta** works:
- For every feature a **shadow feature** is created by randomly shuffling its values (so it has no relation with the target any more).
- A random forest is trained on the real + shadow features, and the importance of each real feature is compared with the **best shadow feature**.
- This is repeated many times. Features that are significantly more important than random noise are **confirmed** (rank 1), features that are clearly worse are **rejected**, and undecided ones are **tentative** (rank 2).

**TASK: Define a `RandomForestClassifier` (`n_jobs=-1`, `class_weight='balanced'`, `max_depth=5`, `random_state=RANDOM_STATE`) and wrap it in a `BorutaPy` feature selector (`n_estimators='auto'`, `max_iter=50`, `random_state=RANDOM_STATE`). Fit the selector on the training set only. Remember to use `.values` to pass numpy arrays. *(This takes about 15-30 seconds.)***

In [ ]:
rf = RandomForestClassifier(
    n_jobs=-1, class_weight='balanced', max_depth=5, random_state=RANDOM_STATE
)

feat_selector = BorutaPy(
    rf, n_estimators='auto', max_iter=50, random_state=RANDOM_STATE, verbose=0
)

start_time = time.time()
feat_selector.fit(X_train.values, y_train.values)
print(f"Boruta fitting completed in {time.time() - start_time:.2f} seconds.")

**TASK: Print the number of selected (confirmed) features and the number of tentative features. *(In the solution Boruta kept 50 of the 64 pixels and found 0 tentative features.)***

In [ ]:
n_confirmed = feat_selector.n_features_
n_tentative = int(np.sum(feat_selector.support_weak_))
n_rejected = int(np.sum(~feat_selector.support_ & ~feat_selector.support_weak_))

print(f"Confirmed features (kept):   {n_confirmed} / {X_train.shape[1]}")
print(f"Tentative features:          {n_tentative}")
print(f"Rejected features (dropped): {n_rejected}")

**TASK: Build a dataframe `feature_ranks` with the columns `feature`, `rank` (`ranking_`) and `keep` (`support_`), sorted by rank. Show the rejected features.**

In [ ]:
feature_ranks = pd.DataFrame({
    'feature': X_train.columns,
    'rank': feat_selector.ranking_,
    'keep': feat_selector.support_
}).sort_values('rank')

rejected_features = feature_ranks[~feature_ranks['keep']]
print(f"Total rejected features: {len(rejected_features)}")
rejected_features

**TASK: Visualise the result: show the average digit of the training set (`X_train.mean()` reshaped to 8 x 8) next to the Boruta mask (`feat_selector.support_` reshaped to 8 x 8, e.g. with `cmap='RdYlGn'`).**

In [ ]:
mean_digit = X_train.mean().values.reshape(8, 8)
boruta_mask = feat_selector.support_.reshape(8, 8).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

im0 = axes[0].imshow(mean_digit, cmap='gray_r')
axes[0].set_title('Average Digit (Training Set Mean)', fontsize=12, fontweight='bold')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].axis('off')

im1 = axes[1].imshow(boruta_mask, cmap='RdYlGn')
axes[1].set_title('Boruta Selection Mask (Green=Keep, Red=Drop)', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, ticks=[0, 1])
axes[1].axis('off')

plt.tight_layout()
plt.show()

**QUESTION: Which pixels did Boruta reject? Does this match your expectation from the standard deviation plot in section 1? Why are these pixels irrelevant?**

*Your answer:*

Boruta rejected 14 pixels in total: the entire leftmost column (`pixel_0_0` through `pixel_7_0`) and most of the rightmost column (`pixel_0_7` through `pixel_5_7`).

This matches the standard deviation plot from section 1. The 3 constant pixels with zero variance (`pixel_0_0`, `pixel_4_0`, and `pixel_4_7`) were among the first to go (rank 12), and the other rejected pixels are their immediate border neighbors that barely have any variance either.

They are irrelevant because digits are drawn inside the central 6x6 area. Almost nobody touches the far outer columns. In Boruta's trials, the random forest found that shuffled shadow noise was just as useful as these border pixels, so Boruta confirmed they carry no real signal.

## 4 - Model Performance: All Features vs Selected Features

**TASK: Create `X_train_sel` and `X_test_sel` that contain only the features selected by Boruta. Then write a helper function `evaluate_model(name, model, X_tr, X_te, y_tr, y_te)` that returns a dictionary with the model name, the number of features, the 5-fold cross-validated accuracy on the training set (`cross_val_score`), the test accuracy and the time needed to fit the model. Create an empty list `results` to store all results.**

In [ ]:
X_train_sel = X_train.loc[:, feat_selector.support_]
X_test_sel = X_test.loc[:, feat_selector.support_]

print(f"X_train_sel shape: {X_train_sel.shape}")
print(f"X_test_sel shape:  {X_test_sel.shape}")

In [ ]:
results = []

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, n_features=None):
    start = time.time()
    model.fit(X_tr, y_tr)
    fit_time = time.time() - start
    
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy')
    y_pred = model.predict(X_te)
    test_acc = accuracy_score(y_te, y_pred)
    
    n_feat = n_features if n_features is not None else X_tr.shape[1]
    
    return {
        'model': name,
        'n_features': n_feat,
        'cv_acc_mean': round(cv_scores.mean(), 4),
        'cv_acc_std': round(cv_scores.std(), 4),
        'test_acc': round(test_acc, 4),
        'fit_time_s': round(fit_time, 4)
    }

**TASK: Evaluate a `RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)` on all features and on the Boruta features, append both results to `results` and show them as a dataframe. *(In the solution we got a test accuracy ≈ 0.967 with all features and ≈ 0.971 with the selected features.)***

In [ ]:
rf_all = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
results.append(evaluate_model("Random Forest (All features)", rf_all, X_train, X_test, y_train, y_test))

rf_sel = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
results.append(evaluate_model("Random Forest (Boruta features)", rf_sel, X_train_sel, X_test_sel, y_train, y_test))

pd.DataFrame(results)

**QUESTION: Compare the two models.**
- Did removing the rejected pixels hurt the performance?
- What are the advantages of a model with fewer features, even if the accuracy stays the same?

*Your answer:*

Dropping the 14 rejected pixels did not hurt the model at all. Test accuracy went slightly up, from 0.9667 (96.7%) on all 64 pixels to 0.9711 (97.1%) on the 50 Boruta pixels. The 5-fold CV score stayed steady at 0.9740.

Even if accuracy were identical, working with 50 features instead of 64 has clear practical advantages:
- Training and inference run faster because each decision tree evaluates fewer potential split candidates at every node.
- There is less risk of overfitting to background quirks, since we stripped out noise columns that could trigger accidental splits.
- In real-world hardware (like a handheld scanner or postal sorter), measuring fewer inputs means simpler image sensors, lower bandwidth, and less storage.

## 5 - Dimension Reduction with PCA

PCA looks for the directions with the **largest variance**. Features with a larger scale would dominate these directions, so (just like in class) we **standardize** the data first. The scaler is fitted on the training set only and then applied to the test set.

**TASK: Standardize the data with a `StandardScaler`: fit it on `X_train` and transform both `X_train` and `X_test` (store them in `X_train_scaled` and `X_test_scaled`). Check that the scaled training features have mean 0 and standard deviation 1.**

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape:  {X_test_scaled.shape}")

**TASK: Fit a `PCA` with all components on `X_train_scaled`. Store the `explained_variance_ratio_` and its cumulative sum (`np.cumsum`) in `explained_variance_ratio` and `cumulative_variance`. Print the explained variance of the first 5 components and the variance explained by the first two components together. *(In the solution PC1 explains ≈ 0.12 and PC1 + PC2 ≈ 0.22 of the variance.)***

In [ ]:
pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_train_scaled)

explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

print("Explained variance ratio of first 5 components:")
for i in range(5):
    print(f"  PC{i+1}: {explained_variance_ratio[i]:.4f} ({explained_variance_ratio[i]*100:.2f}%)")

print(f"\nVariance explained by PC1 + PC2 together: {cumulative_variance[1]:.4f} ({cumulative_variance[1]*100:.2f}%)")

**TASK: Make a figure with two subplots: the Cumulative Variance Plot (with horizontal lines at 90% and 95%) and the Scree Plot (explained variance ratio per component).**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative Variance Plot
axes[0].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', markersize=4, color='#1f77b4', lw=2)
axes[0].axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
axes[0].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[0].set_xlabel('Number of Principal Components', fontsize=11)
axes[0].set_ylabel('Cumulative Explained Variance', fontsize=11)
axes[0].set_title('Cumulative Variance Explained', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='lower right')

# Scree Plot
axes[1].bar(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, color='#348abd', alpha=0.8)
axes[1].set_xlabel('Principal Component Index', fontsize=11)
axes[1].set_ylabel('Explained Variance Ratio', fontsize=11)
axes[1].set_title('Scree Plot (Variance per Component)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**TASK: Determine how many components are needed to explain 80%, 90%, 95% and 99% of the variance. Store the number of components for 95% in `n_components_95`. *(In the solution: 21, 31, 40 and 54 components.)***

In [ ]:
thresholds = [0.80, 0.90, 0.95, 0.99]
components_needed = {}

for thresh in thresholds:
    n_comp = int(np.argmax(cumulative_variance >= thresh) + 1)
    components_needed[int(thresh * 100)] = n_comp
    print(f"{int(thresh * 100)}% variance explained: {n_comp} components (out of 64)")

n_components_95 = components_needed[95]
print(f"\nStored n_components_95 = {n_components_95}")

**QUESTION: Interpret the scree plot and the cumulative variance plot. Is there a clear elbow? How many components would you keep and how much of a reduction is that compared to the 64 original pixels?**

*Your answer:*

The scree plot shows a smooth decline rather than a sharp elbow. PC1 accounts for about 11.9% of the variance, PC2 adds 9.7%, and after that the individual contributions drop off gradually. 

Because there is no single cutoff elbow, a standard rule of thumb like keeping 95% of the variance works well here. That requires 40 components (`n_components_95 = 40`). 

Going from 64 pixels down to 40 components cuts the dimensionality by 37.5% ((64 - 40) / 64) while sacrificing only 5% of the total variance in the data. If we were willing to accept 80% variance, we could compress even further to 21 components (a 67.2% reduction).

**TASK: Transform `X_train_scaled` with the fitted PCA and make a scatter plot of the first two principal components, coloured by the digit (`c=y_train`, `cmap='tab10'`). Add axis labels, a title and a colour bar.**

In [ ]:
X_train_pca = pca.transform(X_train_scaled)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    X_train_pca[:, 0], X_train_pca[:, 1],
    c=y_train, cmap='tab10', alpha=0.7, s=25, edgecolors='none'
)
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.set_label('Digit Label (0-9)', fontsize=11)
plt.xlabel(f'PC1 ({explained_variance_ratio[0]*100:.1f}% variance)', fontsize=11)
plt.ylabel(f'PC2 ({explained_variance_ratio[1]*100:.1f}% variance)', fontsize=11)
plt.title('First Two Principal Components of Digits Dataset', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

**TASK: Each principal component is a linear combination of the 64 pixels. Visualise the loadings (`pca.components_`) of the first three components as 8 x 8 images (use a diverging colour map such as `'coolwarm'`).**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for i in range(3):
    loading_img = pca.components_[i].reshape(8, 8)
    vmax = np.max(np.abs(loading_img))
    im = axes[i].imshow(loading_img, cmap='coolwarm', vmin=-vmax, vmax=vmax)
    axes[i].set_title(f'PC{i+1} Loadings ({explained_variance_ratio[i]*100:.1f}% var)', fontsize=12, fontweight='bold')
    axes[i].axis('off')
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

**QUESTION: Interpret the 2D scatter plot and the loadings.**
- Which digits are well separated with only two components, and which ones overlap?
- PC1 and PC2 together explain only about 22% of the variance. Do you expect that a model on 2 components can classify the digits well?
- Can you still say which original pixels are important after PCA? Compare with Boruta.

*Your answer:*

In the 2D scatter plot, digit 0 stands out clearly on the right, and digit 4 forms its own pocket toward the upper left. But digits 1, 2, 8, and 9 pile on top of each other in the middle. 

I do not expect a model trained on just 2 components to classify digits well. PC1 and PC2 together capture only 21.6% of the variance, throwing away over 78% of the information in the dataset. (And when we test it later in Section 6, 2 components only reaches about 51.1% accuracy).

After PCA, you can no longer say which individual pixels are kept or dropped. Each component is a linear mixture of all 64 pixels (like a smoothie where every fruit is blended in). Boruta is completely different: it either keeps or drops specific original pixels, leaving their identity and meaning intact.

**TASK: Effect of scaling: fit a PCA on the unscaled `X_train` and calculate how many components are needed for 95% of the variance. Compare with the scaled version. *(In the solution: 29 components without scaling vs 40 with scaling.)***

In [ ]:
pca_unscaled = PCA(random_state=RANDOM_STATE)
pca_unscaled.fit(X_train)

cum_var_unscaled = np.cumsum(pca_unscaled.explained_variance_ratio_)
n_comp_unscaled_95 = int(np.argmax(cum_var_unscaled >= 0.95) + 1)

print(f"Components for 95% variance WITHOUT scaling: {n_comp_unscaled_95}")
print(f"Components for 95% variance WITH scaling:    {n_components_95}")

**QUESTION: All pixels are measured on the same scale (0-16). Why do you need a different number of components with and without scaling? Is standardization always necessary before PCA? When is it essential?**

*Your answer:*

Even though every pixel is on the 0 to 16 scale, center pixels vary a lot between images (variances up to 40+), while border pixels barely move. Unscaled PCA chases raw variance, so the high-variance center pixels dominate the first few components, hitting 95% variance in just 29 components. When we standardize, every pixel gets scaled to variance = 1. That levels the playing field across all 64 pixels, so variance is spread out more evenly and it takes 40 components to reach 95%.

Standardization is not always required when all features share the exact same unit and scale, especially if higher variance genuinely reflects greater signal (as in raw pixel brightness). 

However, standardization is mandatory whenever features are measured in different units (like salary in dollars, age in years, and square footage). Without scaling, features with large numbers simply overpower everything else in the covariance matrix, regardless of whether they actually help predict the target.

## 6 - Model Performance on Principal Components

We now train models on the principal components. By putting the scaler, the PCA and the model in a **pipeline**, scaling and PCA are re-fitted inside every cross-validation fold, so no information leaks from the validation folds. We use a **logistic regression**, a linear model that often benefits from PCA, and also a random forest.

**TASK: Build three pipelines with `make_pipeline` and evaluate them with `evaluate_model` (on the unscaled `X_train` / `X_test`, the pipeline does the scaling): (1) `StandardScaler` + `LogisticRegression(max_iter=5000)` on all pixels, (2) `StandardScaler` + `PCA(n_components=n_components_95)` + `LogisticRegression`, (3) `StandardScaler` + `PCA(n_components=n_components_95)` + `RandomForestClassifier(n_estimators=200)`. Set `n_features` to the number of components for the PCA models and append all results. *(In the solution we got test accuracies ≈ 0.978, ≈ 0.960 and ≈ 0.951.)***

In [ ]:
pipe_lr_all = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
)
results.append(evaluate_model("Logistic Regression (All features)", pipe_lr_all, X_train, X_test, y_train, y_test, n_features=64))

pipe_lr_pca = make_pipeline(
    StandardScaler(),
    PCA(n_components=n_components_95, random_state=RANDOM_STATE),
    LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
)
results.append(evaluate_model(f"Logistic Regression (PCA {n_components_95})", pipe_lr_pca, X_train, X_test, y_train, y_test, n_features=n_components_95))

pipe_rf_pca = make_pipeline(
    StandardScaler(),
    PCA(n_components=n_components_95, random_state=RANDOM_STATE),
    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
)
results.append(evaluate_model(f"Random Forest (PCA {n_components_95})", pipe_rf_pca, X_train, X_test, y_train, y_test, n_features=n_components_95))

pd.DataFrame(results)

**TASK: Investigate how the accuracy depends on the number of components: loop over `[2, 5, 10, 15, 20, 30, 40, 50, 64]`, fit the `StandardScaler` + `PCA(n_components=k)` + `LogisticRegression` pipeline and store the test accuracy. Print the results and plot the accuracy against the number of components. *(In the solution: ≈ 0.51 with 2 PCs and ≈ 0.93 with 15 PCs.)***

In [ ]:
components_range = [2, 5, 10, 15, 20, 30, 40, 50, 64]
pca_accuracies = []

print("Accuracy by number of PCA components:")
for k in components_range:
    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=k, random_state=RANDOM_STATE),
        LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
    )
    pipe.fit(X_train, y_train)
    acc = pipe.score(X_test, y_test)
    pca_accuracies.append(acc)
    print(f"  k = {k:2d}: Test Accuracy = {acc:.4f}")

plt.figure(figsize=(8, 4.5))
plt.plot(components_range, pca_accuracies, marker='o', color='#2ca02c', lw=2)
plt.axhline(y=0.9778, color='gray', linestyle='--', label='All 64 Features Baseline (0.978)')
plt.xlabel('Number of Principal Components (k)', fontsize=11)
plt.ylabel('Test Accuracy', fontsize=11)
plt.title('Logistic Regression Accuracy vs Number of Principal Components', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.show()

**QUESTION: Interpret the results of the PCA models.**
- How does the accuracy evolve with the number of components? Where does adding components stop paying off?
- Did PCA improve the performance compared to all features? Why (not)?
- Why is the random forest on PCs worse than the random forest on the original (Boruta) pixels?

*Your answer:*

Accuracy climbs quickly at first: from 51.1% with 2 components up to 92.7% at 15 components, and 94.2% at 20 components. After 30 to 40 components, the curve flattens out (96.0% at 40, peaking at 98.0% at 50). Adding components past 30 gives diminishing returns.

PCA did not meaningfully beat using all features for Logistic Regression (96.0% with 40 components vs 97.8% on all 64 features). This happens because PCA is unsupervised. It preserves maximum variance across the whole dataset, but variance is not the same thing as class separability. Sometimes a direction with lower overall variance holds the exact cue needed to tell two digits apart.

Random Forest did noticeably worse on PCA components (95.1%) than on the original Boruta pixels (97.1%). Decision trees make splits along single axes (pixel_x <= threshold). Original pixels are localized and sparse, which trees split on very naturally. But principal components are dense mixtures of all 64 pixels. To isolate a boundary in that rotated space, trees have to construct an awkward staircase of cuts, which hurts their performance.

## 7 - Comparison of all approaches

**TASK: Combine all results in one dataframe `results_df`, sorted by test accuracy.**

In [ ]:
results_df = pd.DataFrame(results).sort_values('test_acc', ascending=False).reset_index(drop=True)
results_df

**QUESTION: Which approach would you choose in each of the following situations? Motivate your answer.**
- (a) A doctor / business user wants to know **which original variables** drive the prediction.
- (b) You need a **compact, fast** model and interpretability is not important.
- (c) You want to **visualise** a high-dimensional dataset in 2D.

*Your answer:*

(a) **When domain users need to know which real variables drive predictions**: Choose **Boruta (Feature Selection)**. A doctor cannot act on "Component 1 is high"; they need to know if blood pressure, glucose, or a specific biomarker is elevated. Boruta keeps the original measurements untouched.

(b) **When you need a compact, fast model without interpretability constraints**: Choose **PCA + Logistic Regression**. Compressing 64 features into 15 to 30 components still reaches 93% to 95% test accuracy while speeding up matrix math and shrinking the model's memory footprint.

(c) **When you want to visualize high-dimensional data in 2D**: Choose **PCA** (or t-SNE / UMAP). Setting `n_components=2` projects all 64 dimensions onto the two directions of highest variance, letting us see clusters and digit groupings on a single plot.

### Summary

**QUESTION: Summarise what you learned in this exercise. Consider the difference between feature selection and dimension reduction, the effect on model performance and the role of scaling.**

*Your answer:*

Feature selection and dimension reduction attack the same problem from opposite directions. 

Boruta keeps the original features and weeds out the noise. It dropped 14 edge pixels that never had any ink, which kept our feature meanings intact and bumped Random Forest test accuracy from 96.7% to 97.1%. 

PCA takes all 64 pixels and compresses them into brand-new linear components. That works well for linear models like Logistic Regression (96.0% with 40 components), but it hurts tree models. Random Forest dropped from 97.1% on Boruta pixels to 95.1% on PCA components, because trees make orthogonal cuts and struggle with rotated combinations of coordinates.

Scaling is critical for PCA because the algorithm only follows variance. When we did not scale, center pixels dominated the components and reached 95% variance in 29 dimensions; with scaling, it took 40. And just like in Lesson 6, scaling and PCA must sit inside a Pipeline so test data never leaks into the feature transformations.

## Great work!

----